# 01. Data Preparation & Leak-Free Dataset Splitting

**Project:** Road Sign Recognition Using Deep Learning (Cambodian Traffic Signs Dataset - CamTSD)
**Classes:** 26 Selected Traffic Sign Categories

### Objective
This notebook establishes the leak-free data pipeline for the project:
1. Load and inspect raw VIA JSON annotations (`CAM_TSR_v3_json.json`).
2. Filter for the 26 canonical traffic sign classes (excluding `UNDEFINED`).
3. Perform **source-image-level splitting** (70% train, 15% validation, 15% test) to prevent multi-sign parent photo leakage.
4. Extract sign bounding box crops into separated split directories.
5. Save and audit split manifests (`train.csv`, `val.csv`, `test.csv`).
6. Test the PyTorch `TrafficSignDataset` and `DataLoader` pipelines.

## 1. Environment Setup & Reproducibility

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch

# Add project root to sys.path for modular imports
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLASS_NAMES, CLASS_TO_IDX, NUM_CLASSES, get_paths
from src.seed import set_seed
from src.data.prepare_dataset import (
    load_raw_annotations,
    parse_and_filter_records,
    split_source_images,
    crop_and_save_dataset,
    validate_dataset_integrity,
)
from src.data.dataset import TrafficSignDataset, get_dataloaders
from src.data.transforms import denormalize, get_train_transforms, get_eval_transforms

# Set global random seed for full reproducibility
set_seed(42)
paths = get_paths()

print("Project Root:", paths.root_dir)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Total Canonical Classes:", NUM_CLASSES)

## 2. Inspect Raw Annotations & Filter Target Classes

In [ ]:
# Check if raw annotations exist in the environment
if paths.raw_annotations_path.exists():
    annotations = load_raw_annotations(paths.raw_annotations_path)
    print(f"Loaded raw annotations: {len(annotations)} image records")
    
    records, raw_counts = parse_and_filter_records(annotations)
    print(f"Extracted {len(records)} valid sign annotations across {len(raw_counts)} classes.")
    
    # Display class counts
    count_df = pd.DataFrame(list(raw_counts.items()), columns=["Class Name", "Annotation Count"])
    count_df = count_df.sort_values(by="Annotation Count", ascending=False).reset_index(drop=True)
    print("\nClass Distribution Preview (Top 10):")
    print(count_df.head(10).to_string(index=False))
else:
    print(f"Raw annotations not found at {paths.raw_annotations_path}.")
    print("Using existing split manifests if already generated.")

## 3. Leak-Free Source-Image Level Splitting & Dataset Generation

> **Key Methodological Principle:**  
> Because single photographs can contain multiple traffic signs, we partition **original source images** (70% train, 15% val, 15% test) before extracting crops. All crops from the same photograph remain strictly within the same split partition.

In [ ]:
if paths.raw_annotations_path.exists() and paths.raw_images_dir.exists():
    print("Executing source-image splitting and bounding box cropping...")
    split_map = split_source_images(records, seed=42)
    
    all_df = crop_and_save_dataset(
        raw_images_dir=paths.raw_images_dir,
        records=records,
        split_map=split_map,
        processed_dir=paths.processed_dir,
        splits_dir=paths.splits_dir,
    )
    print(f"Successfully generated and saved {len(all_df)} crops.")
else:
    print("Raw image extraction skipped (running in inspection mode).")

## 4. Comprehensive Dataset Split & Integrity Audit

In [ ]:
train_csv = paths.splits_dir / "train.csv"
val_csv = paths.splits_dir / "val.csv"
test_csv = paths.splits_dir / "test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
    
    train_parents = set(train_df["parent_filename"])
    val_parents = set(val_df["parent_filename"])
    test_parents = set(test_df["parent_filename"])
    
    print("=" * 60)
    print("DATASET SPLIT & INTEGRITY AUDIT REPORT")
    print("=" * 60)
    print(f"Source Images -> Train: {len(train_parents):4d} | Val: {len(val_parents):4d} | Test: {len(test_parents):4d} | Total: {len(train_parents | val_parents | test_parents):4d}")
    print(f"Cropped Signs -> Train: {len(train_df):4d} | Val: {len(val_df):4d} | Test: {len(test_df):4d} | Total: {len(train_df) + len(val_df) + len(test_df):4d}")
    print(f"Classes Present in Splits: {len(set(train_df['class_name']) | set(val_df['class_name']) | set(test_df['class_name']))} / 26")
    
    # Overlap validation
    overlap_train_val = len(train_parents.intersection(val_parents))
    overlap_train_test = len(train_parents.intersection(test_parents))
    overlap_val_test = len(val_parents.intersection(test_parents))
    
    print("\n--- Verification Checks ---")
    print(f"[1] Zero Parent-Image Overlap (Train/Val):   {overlap_train_val == 0} (overlap count: {overlap_train_val})")
    print(f"[2] Zero Parent-Image Overlap (Train/Test):  {overlap_train_test == 0} (overlap count: {overlap_train_test})")
    print(f"[3] Zero Parent-Image Overlap (Val/Test):    {overlap_val_test == 0} (overlap count: {overlap_val_test})")
    print(f"[4] 'UNDEFINED' Label Excluded:              {'UNDEFINED' not in train_df['class_name'].values}")
    print(f"[5] Accidental 'splits' Class Bug Excluded:  {'splits' not in train_df['class_name'].values}")
    print("=" * 60)
else:
    print("Split CSV files not found. Please run dataset preparation pipeline.")

## 5. Class Distribution Across Splits

In [ ]:
if train_csv.exists() and val_csv.exists() and test_csv.exists():
    split_dist = pd.DataFrame({
        "Train": train_df["class_name"].value_counts(),
        "Val": val_df["class_name"].value_counts(),
        "Test": test_df["class_name"].value_counts(),
    }).fillna(0).astype(int)
    
    split_dist["Total"] = split_dist.sum(axis=1)
    split_dist = split_dist.sort_values(by="Total", ascending=False)
    
    plt.figure(figsize=(14, 6))
    split_dist[["Train", "Val", "Test"]].plot(kind="bar", stacked=True, figsize=(14, 6), colormap="viridis")
    plt.title("Class Distribution Across Train, Validation, and Test Splits", fontsize=14)
    plt.xlabel("Traffic Sign Class", fontsize=12)
    plt.ylabel("Number of Images", fontsize=12)
    plt.xticks(rotation=75, ha="right", fontsize=9)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

## 6. PyTorch Dataset & DataLoader Verification

In [ ]:
if train_csv.exists():
    train_loader, val_loader, test_loader = get_dataloaders(paths=paths, batch_size=32, num_workers=0)
    
    print(f"Training Batches:   {len(train_loader)}")
    print(f"Validation Batches: {len(val_loader)}")
    print(f"Test Batches:       {len(test_loader)}")
    
    # Fetch a single batch from training loader
    images, labels = next(iter(train_loader))
    print("\n--- Batch Tensor Shapes ---")
    print(f"Batch Image Tensor Shape: {images.shape} (Expected: [32, 3, 224, 224])")
    print(f"Batch Label Tensor Shape: {labels.shape} (Expected: [32])")
    print(f"Image Data Type:          {images.dtype}")
    print(f"Label Data Type:          {labels.dtype}")
    print(f"Batch Label Range:        Min={labels.min().item()}, Max={labels.max().item()}")
    
    # Display sample cropped signs with denormalization
    images_denorm = denormalize(images[:8]).cpu()
    
    plt.figure(figsize=(14, 7))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        img = images_denorm[i].permute(1, 2, 0).numpy()
        plt.imshow(img)
        plt.title(CLASS_NAMES[labels[i].item()], fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()